In [ ]:
# !pip install surprise

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

mpl.rc('font', family='NanumGothic') # 폰트 설정
mpl.rc('axes', unicode_minus=False) # 유니코드에서 음수 부호 설정

# 차트 스타일 설정
sns.set(font="NanumGothic", rc={"axes.unicode_minus":False}, style='darkgrid')
plt.rc("figure", figsize=(10,8))

In [ ]:
def label_at(X_data, li):
  # 라벨인코딩
  from sklearn.preprocessing import LabelEncoder

  for i in li:
    encoder = LabelEncoder()
    X_train = list(X_data[f'{i}'].unique())

    # X_train데이터를 이용 피팅하고 라벨숫자로 변환한다
    encoder.fit(X_train)
    X_encoded = encoder.fit_transform(X_data[f'{i}'])
    X_data[f'{i}_le'] = list(X_encoded)
    X_data = X_data.drop([f'{i}'], axis=1)
  return X_data, encoder

In [2]:
import surprise
print(surprise.__version__)

1.1.3


### 데이터 로드

In [5]:
%cd /content/drive/MyDrive/AI응용_팀플_Lpoint

/content/drive/MyDrive/AI응용_팀플_Lpoint


In [6]:
df = pd.read_csv('./fin_data_set.csv')
df1 = pd.read_csv('./final_data.csv')
age_cluster = pd.read_csv('./age_cluster.csv')
fre_cluster = pd.read_csv('./fre_cluster.csv')

In [9]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 723242 entries, 0 to 723241
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   cno        723242 non-null  int64 
 1   gender     723242 non-null  object
 2   age_grp    723242 non-null  object
 3   scls_c_nm  723242 non-null  object
 4   Recency    723242 non-null  int64 
 5   Frequency  723242 non-null  int64 
 6   Monetory   723242 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 38.6+ MB


### 한웅님코드

In [24]:
age_df = age_cluster[['cluster','cno']]
X_data = df1.merge(age_df, on='cno')
X_data.drop(['gender','age_grp'],axis=1,inplace=True)
X_data, le_encoder = label_at(X_data, ['scls_c_nm'])


In [61]:
def score_dataset(df):
  df_02_lp_filter = df[['cno','scls_c_nm_le','Recency','Frequency','Monetory']]

  # RFM별 10분위수 기준 테이블 생성
  rng = np.arange(0.1,1.1,0.1)
  col_list = df_02_lp_filter.columns
  RFM_list= col_list[2:]
  df_03_RFM_quan = pd.DataFrame()

  quan = []
  for i in RFM_list:
    for j in rng:
      quan.append(df_02_lp_filter[i].quantile(q=j))
      #print("{} {}% 분위수: {}".format(i,int(j*100),int(df_05_merge_RFM_Score[i].quantile(q=j))))      
    df_03_RFM_quan = pd.concat([df_03_RFM_quan,pd.DataFrame(quan, columns=[i])],axis = 1)
    quan = []

  df_03_RFM_quan = pd.concat([pd.DataFrame(rng, columns=['quantile']), df_03_RFM_quan], axis = 1)


  # 각 분위수별 SCORE 산출
  df_03_RFM_quan['Recency_Score'] = df_03_RFM_quan['Recency'].rank(method='min', ascending=False)
  df_03_RFM_quan['Frequency_Score'] = df_03_RFM_quan['Frequency'].rank(method='min', ascending=True)
  df_03_RFM_quan['Monetory_Score'] = df_03_RFM_quan['Monetory'].rank(method='min', ascending=True)


  # RFM 범위 컬럼 생성
  ## 어떤 고객의 Recency 값이 0(Recency_start) ~ 236(Recency) 사이에 있다면 Recency_Score를 10으로 본다,
  df_03_RFM_quan['Recency_start'] = (df_03_RFM_quan['Recency'].shift(1)+1).fillna(0)
  df_03_RFM_quan['Frequency_start'] = (df_03_RFM_quan['Frequency'].shift(1)).fillna(0)
  df_03_RFM_quan['Monetory_start'] = (df_03_RFM_quan['Monetory'].shift(1)+1).fillna(0)
  df_04_RFM_quan_fin = df_03_RFM_quan[['quantile','Recency_start','Recency','Frequency_start','Frequency','Monetory_start','Monetory','Recency_Score','Frequency_Score','Monetory_Score']]

  df_02_lp_filter.reset_index(drop=True,inplace=True)

  # 각 고객별  Recency_SCORE Frequency_SCORE, Monetory_SCORE 산출
  rnk = []
  df_05_calcul_RFM_Score = df_02_lp_filter

  for i in RFM_list:
    for j in range(len(df_02_lp_filter)):
      for k in range(10):
        if ( df_02_lp_filter[i][j] >= df_04_RFM_quan_fin[f'{i}_start'][k] ) and ( df_02_lp_filter[i][j] <= df_04_RFM_quan_fin[i][k] ):
          rnk.append(df_04_RFM_quan_fin[f'{i}_Score'][k])
          break
    print(f'{i} row수: ',len(rnk))
    df_05_calcul_RFM_Score = pd.concat([df_05_calcul_RFM_Score,pd.DataFrame(rnk, columns=[f'{i}_Score'])],axis = 1)
    rnk = []

  # RFM Score 생성
  # 각각의 Score에 임의의 가중치를 곱하여 최종 점수 생성
  df_05_calcul_RFM_Score['RFM_SCORE'] = (df_05_calcul_RFM_Score['Recency_Score'] * 0.4) + (df_05_calcul_RFM_Score['Frequency_Score'] * 0.3) + (df_05_calcul_RFM_Score['Monetory_Score'] * 0.3)

  # 행렬 분해에 필요없는 컬럼 제거
  df_06_RFM_fin_tbl = df_05_calcul_RFM_Score.drop(labels=['Recency','Frequency','Monetory','Recency_Score','Frequency_Score','Monetory_Score'],axis=1)
  df_06_RFM_fin_tbl = df_06_RFM_fin_tbl.rename(columns = {'cno':'Customer','scls_c_nm_le':'Product'}) 
  return df_06_RFM_fin_tbl


In [62]:
from surprise import Dataset
from surprise import Reader
from surprise.model_selection import train_test_split

X1 = X_data[X_data['cluster']==0]
X2 = X_data[X_data['cluster']==1]
X3 = X_data[X_data['cluster']==2]
X4 = X_data[X_data['cluster']==3]
X5 = X_data[X_data['cluster']==4]
X6 = X_data[X_data['cluster']==5]


# SVD 알고리즘의 하이퍼파라미터를 변경해가며 rmse 값을 최저치로 낮추는 튜닝 수행
from surprise import SVD
from surprise.model_selection import GridSearchCV

rmse_score = []
acc = []
param_grid = {'n_factors': [50, 100], 'lr_all': [0.02,0.01], 'reg_all':[0.02,0.01], 'n_epochs':[40, 50]}
gs = GridSearchCV(algo_class = SVD, measures=['RMSE'], param_grid=param_grid)
i = 1

for df in [X1, X2, X3, X4, X5, X6]:  
  final_dataset = score_dataset(df)
  reader = Reader(rating_scale=(final_dataset['RFM_SCORE'].min(), final_dataset['RFM_SCORE'].max())) 
  data = Dataset.load_from_df(final_dataset[['Customer', 'Product', 'RFM_SCORE']],reader) 

  train_data, test_data = train_test_split(data, test_size=0.25, random_state=0)

  gs.fit(data)
  print(gs.best_params['rmse'])

  # 최적의 파라미터로 학습 및 예측 
  from surprise import accuracy

  best_params = gs.best_params['rmse']
  algo = SVD(n_factors = best_params['n_factors'],lr_all = best_params['lr_all'], reg_all= best_params['reg_all'], n_epochs=best_params['n_epochs'])
  algo.fit(train_data)
  prediction = algo.test(test_data)
  RMSE = accuracy.rmse(prediction)
  
  print(f'{i}군집 RMSE : ', RMSE)
  i += 1
  rmse_score.append(RMSE)

  # 예측 결과 데이터 프레임으로 변환
  col = ['Customer', 'Product', 'RFM_SCORE', 'PRED_SCORE', 'DETAILS']
  res = pd.DataFrame(prediction,columns = col)
  res.drop(labels=['DETAILS'],axis=1,inplace=True)

  res['abs_diff'] = round(abs(res['RFM_SCORE'] - res['PRED_SCORE']),1)

  # 학습 모델의 평가 지표 (정확도, RMSE) 출력
  res['score_diff'] = abs(res['PRED_SCORE'] - res['RFM_SCORE'])
  md_acc = round(1 - (sum(res['score_diff']) / sum(res['PRED_SCORE'])),2) * 100
  acc.append(md_acc)
  print(f'학습 모델 정확도: {md_acc}%')
  print(f'학습 모델 RMSE: {round(RMSE,2)}')

Recency row수:  99569
Frequency row수:  99569
Monetory row수:  99569
{'n_factors': 100, 'lr_all': 0.02, 'reg_all': 0.02, 'n_epochs': 50}
RMSE: 2.2678
1군집 RMSE :  2.267801150989199
학습 모델 정확도: 65.0%
학습 모델 RMSE: 2.27
Recency row수:  110276
Frequency row수:  110276
Monetory row수:  110276
{'n_factors': 100, 'lr_all': 0.02, 'reg_all': 0.02, 'n_epochs': 50}
RMSE: 2.2629
2군집 RMSE :  2.2629094839438277
학습 모델 정확도: 65.0%
학습 모델 RMSE: 2.26
Recency row수:  126936
Frequency row수:  126936
Monetory row수:  126936
{'n_factors': 100, 'lr_all': 0.02, 'reg_all': 0.02, 'n_epochs': 50}
RMSE: 2.2796
3군집 RMSE :  2.2796486436148675
학습 모델 정확도: 64.0%
학습 모델 RMSE: 2.28
Recency row수:  83744
Frequency row수:  83744
Monetory row수:  83744
{'n_factors': 100, 'lr_all': 0.02, 'reg_all': 0.02, 'n_epochs': 50}
RMSE: 2.2823
4군집 RMSE :  2.282282830399455
학습 모델 정확도: 65.0%
학습 모델 RMSE: 2.28
Recency row수:  122296
Frequency row수:  122296
Monetory row수:  122296
{'n_factors': 100, 'lr_all': 0.02, 'reg_all': 0.02, 'n_epochs': 50}
RMSE: 2.276

In [56]:
# 학습 모델의 평가 지표 (정확도, RMSE) 출력
res['score_diff'] = abs(res['PRED_SCORE'] - res['RFM_SCORE'])
md_acc = round(1 - (sum(res['score_diff']) / sum(res['PRED_SCORE'])),2) * 100
print(f'학습 모델 정확도: {md_acc}%')
print(f'학습 모델 RMSE: {round(RMSE,2)}')

학습 모델 정확도: 65.0%
학습 모델 RMSE: 2.26


In [ ]:
# 타겟 고객 필터링
t_cust_filter = df_06_RFM_fin_tbl.groupby(['Customer'])\
                       .agg({"Product":"count"}) \
                       .rename(columns = {'Product':'PROD_CNT'}) \
                       .sort_values(by=['PROD_CNT'], ascending=[False]) \
                       .reset_index() 
t_cust_filter

In [ ]:
# 타겟 고객이 구매하지 않은 상품 리스트를 출력하는 함수

def get_unbuy_surprise(ratings, userId):
    #입력값으로 들어온 userId에 해당하는 고객의 평점이 존재하는 모든 상품을 리스트로 생성
    buy_prod = ratings[ratings['Customer']== userId]['Product'].tolist()
     
    # 모든 상품을 리스트로 생성.
    total_prod = ratings['Product'].drop_duplicates().tolist()
     
    # 모든 상품들 중 이미 평점이 존재하는 상품을 제외하여 리스트로 생성
    unbuy_prod = [prod for prod in total_prod if prod not in buy_prod]
    print(f' {userId} 고객이 구매한 (RFM Score가 존재하는) 상품 수:',len(buy_prod),'\n'\
          ,f'{userId} 고객이 구매하지 않은 (추천 대상) 상품 수:',len(unbuy_prod),'\n' \
          ,'롯데 백화점의 전체 상품 수:',len(total_prod))
     
    return unbuy_prod

userId = t_cust_filter.Customer[0]
unbuy_prod = get_unbuy_surprise(df_06_RFM_fin_tbl, userId)

In [ ]:
# 타겟 고객이 구매하지 않은 상품에 대해 예측 점수 생성 및 TOP-10 상품을 추출하는 함수

def recomm_prod_by_surprise(algo, userId, unbuy_prod, top_n=10):
    # 알고리즘 객체의 predict() 메서드를 평점이 없는 상품에 반복 수행한 후 결과를 list 객체로 저장
    predictions = [algo.predict(str(userId), str(prod)) for prod in unbuy_prod]
     
    # predictions list 객체는 surprise의 Predictions 객체를 원소로 가지고 있음.
    # [Prediction(uid='9', iid='1', est=3.69), Prediction(uid='9', iid='2', est=2.98),,,,]
    # 이를 est 값으로 정렬하기 위해서 아래의 sortkey_est 함수를 정의함.
    # sortkey_est 함수는 list 객체의 sort() 함수의 키 값으로 사용되어 정렬 수행.
    def sortkey_est(pred):
        return pred.est
     
    # sortkey_est( ) 반환값의 내림 차순으로 정렬 수행하고 top_n개의 최상위 값 추출.
    predictions.sort(key=sortkey_est, reverse=True)
    top_predictions= predictions[:top_n]
     
    # top_n으로 추출된 상품의 정보 추출. 추천 예상 평점, 상품 이름 추출
    top_prod_ids = [ pred.iid for pred in top_predictions]
    top_prod_rating = [ pred.est for pred in top_predictions]
    top_prod_preds = [(id, rating) for id, rating in zip(top_prod_ids, top_prod_rating)]
    return top_prod_preds
 
#unseen_videos = get_unseen_surprise(ratings, userId)
top_prod_preds = recomm_prod_by_surprise(algo, userId, unbuy_prod, top_n=10)
print(f'##### < {userId} > 고객에게 추천 할만한 상품 Top-10 #####')

for top_prod in top_prod_preds:
    print(top_prod[0], ":", round(top_prod[1],1))

### RFM analysis